# Phonon 31M English — streaming TTS on a free T4 or CPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/phonon-31m-en-streaming-tts.ipynb)

**Runtime: free Colab T4 or CPU. This is the contrast to the [Drama 3 notebook](Drama-3-tts-voice-clone.ipynb), which needs a Colab Pro A100.**

[`gradium/phonon-31m-en`](https://huggingface.co/gradium/phonon-31m-en) is Gradium's 31M English streaming text-to-speech model (Apache 2.0). A flow-matching transformer predicts 24 kHz speech one 80 ms frame at a time; the Mimi decoder turns those latents into audio while later frames are still being generated. The q8 GGUF is **64.5 MB**. The published runtime is [`ptts`](https://pypi.org/project/ptts/) (Rust, no PyTorch), so VRAM is unused even if a T4 is attached.

This notebook loads **`phonon-31m-en` only**. Do not switch the repo ID to `phonon-40m-*`. Those checkpoints are [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/), gated, and not Apache.

**Licence:** Apache 2.0 on `phonon-31m-en` (weights, tokenizer, and the bundled default voice). `ptts` itself is MIT OR Apache-2.0. The 40m family is a different licence and is not used here.

**Runtime:** *Runtime > Change runtime type > T4 GPU* if you have it, or leave the CPU runtime. `ptts` 0.4.2's Linux wheel reports `available_devices() == ["cpu"]`. A T4 is fine and not required. The first run downloads the 64.5 MB q8 file plus the Easton voice from [`gradium/phonon-voices`](https://huggingface.co/gradium/phonon-voices).

**Links:** [model card](https://huggingface.co/gradium/phonon-31m-en) · [ptts (PyPI)](https://pypi.org/project/ptts/) · [runtime](https://github.com/gradium-ai/xn-ptts) · [voices](https://huggingface.co/gradium/phonon-voices) · [Drama 3 notebook](Drama-3-tts-voice-clone.ipynb)


## 1. Install `ptts`

The model card's Python path: `pip install ptts`, then `ptts.TTS("hf://gradium/phonon-31m-en", lang="en", quant="q8")`. The wheel is a Rust extension. It does not replace Colab's torch, so there is no restart step.


In [ ]:
%pip install -q "ptts==0.4.2"

# 0.4.2 is the release that matches the card's Python API (TTS, stream, save).
# The wheel has no weights. phonon-31m-en is downloaded on first TTS() below.


## 2. Runtime check

CPU is enough. If a GPU is attached it is printed and then ignored: this `ptts` build has no CUDA device.


In [ ]:
import platform

import ptts

MODEL_ID = "hf://gradium/phonon-31m-en"
if MODEL_ID != "hf://gradium/phonon-31m-en":
    raise SystemExit("This notebook only loads gradium/phonon-31m-en (Apache 2.0).")

info = ptts.build_info()
print("ptts", ptts.__version__)
print("build", info)
print("devices", ptts.available_devices(), "| threads", ptts.get_num_threads())
print("host", platform.machine(), platform.processor() or "")

try:
    import torch

    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(
            "GPU present (unused by ptts):",
            torch.cuda.get_device_name(),
            f"{props.total_memory / 1024**3:.1f} GB",
        )
    else:
        print("GPU: none — CPU runtime is enough")
except ImportError:
    print("GPU: none — CPU runtime is enough")


## 3. Load `phonon-31m-en`

Same constructor as the [model card](https://huggingface.co/gradium/phonon-31m-en#python). `lang="en"` picks English text normalization (numbers, currency, abbreviations). `quant="q8"` loads `model.q8.gguf` and reports as `q8_0`.


In [ ]:
tts = ptts.TTS(MODEL_ID, lang="en", quant="q8")

print("model", MODEL_ID)
print("device", tts.device, "| quant", tts.quant, "| sample_rate", tts.sample_rate)
print("languages", tts.languages)
print("checkpoint voices", tts.voices)


## 4. Stream a paragraph and measure TTFA

`tts.stream` yields float32 PCM as each 80 ms frame is decoded. Time-to-first-audio (TTFA) is the wall time from entering the stream until the first chunk arrives. The card's published TTFB on a MacBook Air M5 is 5–6 ms; a Colab CPU is slower. The first use of a catalog voice downloads that voice, so Easton is warmed below before the timed run.


In [ ]:
import time

import numpy as np

VOICE = "Easton"
PARAGRAPH = (
    "Hello from Phonon thirty-one million, a streaming text-to-speech model "
    "that runs on a free Colab T4 or even a CPU. "
    "It writes twenty-four kilohertz speech one frame at a time, so the first "
    "audio can start before the rest of this paragraph is ready. "
    "The English weights are Apache two and about sixty-five megabytes, "
    "which is why they fit here when Drama three does not."
)

# Catalog voices download on first use. Warm so TTFA is generation, not the Hub fetch.
tts.synth("Ready.", voice=VOICE)

chunks = []
ttfa_s = None
t0 = time.perf_counter()
with tts.stream(PARAGRAPH, voice=VOICE) as audio:
    for pcm in audio:
        if ttfa_s is None:
            ttfa_s = time.perf_counter() - t0
        chunks.append(np.asarray(pcm, dtype=np.float32))
elapsed_s = time.perf_counter() - t0

if not chunks:
    raise SystemExit("The stream ended with no audio chunks.")

pcm = np.concatenate(chunks)
sample_rate = tts.sample_rate
audio_s = len(pcm) / sample_rate
first_ms = 1000.0 * len(chunks[0]) / sample_rate
print("voice:", VOICE)
print(f"TTFA: {ttfa_s * 1000:.1f} ms")
print(f"first chunk: {len(chunks[0])} samples ({first_ms:.0f} ms at {sample_rate} Hz)")
print(
    f"wall: {elapsed_s:.2f} s | audio: {audio_s:.2f} s | "
    f"realtime: {audio_s / elapsed_s:.1f}x | chunks: {len(chunks)}"
)


## 5. Play and save the WAV

The streamed PCM is written as a mono 16-bit WAV (the same format `tts.save` writes) and played in the notebook. The second cell is the card's one-shot `save` for a short line.


In [ ]:
import os
import wave

from IPython.display import Audio, display

WAV_PATH = "phonon-31m-en-paragraph.wav"
pcm16 = (np.clip(pcm, -1.0, 1.0) * 32767.0).astype("<i2")
with wave.open(WAV_PATH, "wb") as wav:
    wav.setnchannels(1)
    wav.setsampwidth(2)
    wav.setframerate(sample_rate)
    wav.writeframes(pcm16.tobytes())

print(WAV_PATH, os.path.getsize(WAV_PATH), "bytes")
display(Audio(WAV_PATH))


In [ ]:
hello_s = tts.save("phonon-31m-en-hello.wav", "Hello from Phonon.", voice=VOICE)
print("phonon-31m-en-hello.wav", f"{hello_s:.2f}s")
display(Audio("phonon-31m-en-hello.wav"))


## 6. Caveats

- **Apache 2.0 covers `phonon-31m-en` only.** The 40m English, French, German, Spanish, Portuguese, and multilingual checkpoints are CC BY-NC 4.0 and gated. This notebook does not load them.
- **Drama 3 does not fit free Colab.** [`fishaudio/drama-3-int4`](https://huggingface.co/fishaudio/drama-3-int4) is a 35B-total MoE with 20.8 GB of int4 engine weights. The [Drama 3 notebook](Drama-3-tts-voice-clone.ipynb) needs a Colab Pro A100 (40 GB or larger) and GPTQ-Marlin (compute capability 8.0+). A T4 is sm_75 and cannot run that path. Phonon 31M is the free-runtime TTS notebook in this repo.
- **`ptts` is CPU here.** Attaching a T4 does not move inference onto the GPU. Peak working set is the 64.5 MB q8 file plus a small decode buffer, not gigabytes of VRAM.
- **English only.** Text in other languages is read with the wrong pronunciation. Other languages in the Phonon family need a 40m checkpoint, which this notebook will not load.
- **Long inputs are split by sentence**, so prosody does not carry across the whole text.
- **Responsible use.** Do not impersonate a real person, deceive listeners about who is speaking, or clone a voice without consent. Tell listeners when speech is synthetic where that matters.
